# Module 3: The agent loop, stop conditions and turn control

> **Prototype notebook.** It walks this module's steps with the evidence kept on the page and links
> straight to the right place in MLflow and the OpenShift console. The wording is a working draft;
> the module's published text is the lab author's.

In [ ]:
import labnb as lab
%cd ../agent

## Start from module 3

This puts the agent in the state module 3 starts from. Any edits you made are copied to
`agent/.reset-backup/` first, so nothing you wrote is lost.

In [ ]:
!python reset.py 3

Module 3 starts on endpoint B (module 2's repoint is applied), so tool calls now fire.

## 1. Watch it loop

The question needs two steps: `find_part` for the pump, then `check_stock` with the part number it
returns.

In [ ]:
!python agent.py --q m3
lab.evidence(columns=("turns", "ended by", "tools run", "answer"))

Note how many turns it took and how the run ended, compared with module 2.

## 2. Find what went wrong, from the evidence

Open the trace. Count the `model_call_N` spans, and read the tool spans in order.

In [ ]:
lab.trace_link()
lab.tool_results()

The same tool, with the same arguments, on every turn, until the cap in `config.toml` (`max_turns`)
stopped it. The model isn't repeating itself for no reason: something is wrong with what the loop
gives back to it.

Open [`agent.py`](../agent/agent.py) and read `run_agent`. Find two defects: one in what happens to
a tool's result, and one in how the loop decides to stop.

<details><summary>Check your answer</summary>

1. The tool result is fetched and never appended to the message list, so the model never sees it and
   asks again.
2. The loop exits on a turn counter and a text check ("final answer"), and never reads the model's
   own stop condition, `finish_reason`.
</details>

## 3. Fix the loop

Apply the supplied fix, then read what changed:

In [ ]:
!cp solutions/module-03/agent.py .
!git diff --stat -- agent.py
!git diff -- agent.py | head -80

Each tool result is now appended with the `tool_call_id` it answers, and the loop branches on
`finish_reason`: continue on `tool_calls`, return on `stop`. The turn cap stays, as a safety net.

```mermaid
flowchart TD
  Q["The question"] --> C["Call the model"]
  C --> D{"finish_reason?"}
  D -- "tool_calls" --> R["Run each tool it asked for"]
  R --> AP["Append each result, with the tool_call_id it answers"]
  AP --> C
  D -- "stop" --> ANS["Return the model's answer"]
  C -. "safety net" .-> CAP["Turn cap reached: stop and say so"]
```

## 4. Confirm it stops

In [ ]:
!python agent.py --q m3
lab.evidence(columns=("turns", "ended by", "tools run", "answer"))
lab.trace_link()

**This is the module's success signal:** the run ended on the model's own stop condition, well under
the cap. In the trace: `find_part` once, then `check_stock` for each part it returned (the pump and
its seal kit come back together, so expect two `check_stock` calls in one turn), and every result
in the messages the model saw.